# Caso 1. Lluvia: ITCHI con el máximo en P99 (v0.2.0) y en P95, en malla y por municipio

**Qué se compara.** Dos versiones de ITCHI que solo difieren en la lluvia:

| Experimento | Sin escalada | Con suelo saturado (R24 ≥ P95 de 24 h y A72 ≥ P75 de 72 h) | Configuración |
|---|---|---|---|
| v0.2.0 | P75 → 0, P90 → 0.25, P95 → 0.5, **P99 → 1** | P95 → 0.8 | `configs/experiments/v021/v020.yaml` |
| P95 máximo | P75 → 0, P90 → 0.25, **P95 → 1** | **P90 → 0.8** | `configs/experiments/v021/p95.yaml` |

La escalada es la misma (peso f = w(R24: P90 → P95 de 24 h) · w(A72: 0 → P75 de 72 h)); con el máximo en P95 actúa sobre el nodo de abajo, el P90, porque el P95 ya vale 1. El viento es el de v0.2.0 en las dos, así que toda diferencia viene de la lluvia. El notebook 11 (A2, A3) muestra la forma de las rampas.

**Casos.** Emily 2005, Alex 2010, Odile 2014, Patricia 2015 y Cristóbal 2020. (Odile es de 2014: no hubo un Odile que afectara a México en 2012.) Se usan solo las posiciones cuya región de atribución alcanza algún píxel terrestre de México.

**Malla y municipio.** La malla es la de MSWEP (0.1°). El valor municipal es el máximo de ITCHI sobre los píxeles del municipio (notebook 11, A8). Los mapas muestran la clase del máximo del evento (ITCHI_max).

**Qué se espera antes de ver los mapas** (de A2 y A3, para no interpretar a posteriori):
1. P95 nunca baja el peligro: Δclase ≥ 0 en todo píxel y municipio.
2. Sin escalada, todo lo que supera el P95 de 6 h pasa a clase 5 y lo que está entre P90 y P95 sube hasta dos clases.
3. Donde hubo persistencia y suelo saturado (núcleo de lluvia de varios días), la clase 5 empieza ya en el P90, así que esa zona crece más.
4. La clase 4 se estrecha (de ~9 a ~2.4 mm de P6 en la mediana de México): pocos píxeles quedan en ella.

**Cómo se usan las declaratorias.** Solo como prueba ordinal (plan v0.3, P2): en cada versión, la fracción de municipios con declaratoria debe crecer con la clase. No se calibra ni se elige la versión con ellas. El Bloque 1 (`docs/experimento_b1.md`, sección 9) ya evaluó el P95 como máximo con todos los pares municipio–ciclón y no lo adoptó: no mejoró la discriminación y subió un 61 % las alertas de clase 5.

**Entradas.** Hay que correr antes, en `scripts/`, `python casos_v021.py`. Guarda los campos de todos los casos en `outputs/diagnostics/avance_v021/`; este notebook los lee.

## 0. Configuración y datos

In [ ]:
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from itchi.config import get_project_root

ROOT = get_project_root()
sys.path.insert(0, str(ROOT / "scripts"))
import casos_v021 as cv       # noqa: E402
import figuras_v021 as F      # noqa: E402

CFG = cv.ConfigV021()          # rutas (D:/MSWEP_V316_6H, climatologías, IBTrACS, ROCLOUD, municipios)
FORZAR = False                 # True: recalcula aunque exista el caché de outputs/diagnostics/avance_v021/
MODO_MUNICIPIO = "poligonos"   # "poligonos" (shapefile) o "malla" (cada píxel con el valor de su municipio)
FIG = CFG.fig_dir / "caso1_lluvia"
FIG.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 30)


def guardar(fig, nombre):
    fig.savefig(FIG / f"{nombre}.png", dpi=200, bbox_inches="tight")
    plt.show()


assert CFG.mswep_6h_dir.is_dir(), f"no existe {CFG.mswep_6h_dir}"
for d, f in CFG.clim.items():
    assert f.exists(), f"falta {f.name}"

In [ ]:
CASOS = list(cv.CASOS_1_2)                 # Emily 2005, Alex 2010, Patricia 2015, Cristóbal 2020, Odile 2014
DATA = {c: cv.load_or_compute(c, CFG, force=FORZAR) for c in CASOS}
DECL = cv.load_declaratorias(CFG)
for c, C in DATA.items():
    ds = C["ds"]
    print(f"{c}: {ds.sizes['time']} posiciones que alcanzan México "
          f"({pd.Timestamp(ds.time.values[0]):%Y-%m-%d %H} a {pd.Timestamp(ds.time.values[-1]):%Y-%m-%d %H} UTC), "
          f"Vmax máx {float(ds.track_vmax_kt.max()):.0f} kt")

In [ ]:
EXP = ["v020", "p95"]
REF = "v020"

## 1. Los percentiles de 6 h sobre México

Dónde está el nuevo máximo (P95) y qué tanto del eje de lluvia se deja de distinguir (P99 − P95).

**Cómo leerla.**
- **P95.** El umbral de la clase 5 sin escalada con la versión P95. En la vertiente del Pacífico y el sureste suele pasar de 30 mm en 6 h; en el altiplano y el noroeste es mucho menor, así que ahí la clase 5 se alcanza con lluvias modestas en mm, aunque raras para el lugar.
- **P99 − P95.** El tramo de lluvia que en v0.2.0 separaba las clases 4 y 5 y que con P95 queda todo en clase 5.

In [ ]:
guardar(F.fig_percentiles(DATA["CRISTOBAL2020"]), "percentiles_6h_mexico")

## 2. Mapas del evento: malla y municipio

Para cada caso, una figura de 2 × 3:
- **fila de arriba, malla:** clase de ITCHI_max con máximo en P99 (v0.2.0), con máximo en P95, y Δclase (P95 − P99);
- **fila de abajo, municipio:** lo mismo con el máximo municipal.

La línea negra es la trayectoria; los puntos, las posiciones usadas.

**Cómo leerla.**
- **Δclase.** Solo puede ser 0 o positiva (expectativa 1). Un valor negativo indica un error en los campos.
- **Dónde aparece +1 o +2.** Lluvia entre P90 y P99 (expectativa 2). Donde además hubo suelo saturado el cambio es mayor (expectativa 3); el panel b) del notebook 14 muestra cómo revisar R24 y A72 en un píxel.
- **Malla frente a municipio.** El municipio hereda el píxel más alto, así que el mapa municipal es más «rojo». Un municipio grande con un solo píxel en clase 5 aparece completo en clase 5.

In [ ]:
for c, C in DATA.items():
    fig = F.fig_mapa_evento(C, EXP, ref=REF, mode=MODO_MUNICIPIO,
                            title=f"{c}: clase de ITCHI_max, máximo de lluvia en P99 (v0.2.0) frente a P95")
    guardar(fig, f"{c.lower()}_evento_p99_p95")

## 3. El peligro por lluvia solo (H_P_max)

Máximo del evento de H_P en malla, para las dos versiones; los contornos negros son los límites de clase (0.2, 0.4, 0.6, 0.8).

**Cómo leerla.** La diferencia entre paneles es la única fuente del Δclase de la sección 2. Si en el panel P95 los contornos de 0.6 y 0.8 casi coinciden, es el estrechamiento de la clase 4 (expectativa 4).

In [ ]:
for c, C in DATA.items():
    guardar(F.fig_mapa_campo(C, "H_P", EXP, cmap="YlGnBu"), f"{c.lower()}_HP_p99_p95")

## 4. Una posición: el pico del evento sobre México

La posición con mayor ITCHI acumulado sobre tierra. Muestra el campo instantáneo (no el máximo del evento), con los radios de esa posición: R_direct (negro), R_W06 (naranja) y radio de atribución (magenta).

In [ ]:
for c, C in DATA.items():
    t = cv.peak_time(C, "v020")
    guardar(F.fig_mapa_posicion(C, t, EXP, mode=MODO_MUNICIPIO), f"{c.lower()}_posicion_{t:%Y%m%d%H}_p99_p95")

## 5. Conteos por clase

Número de municipios (y de píxeles terrestres) en cada clase del máximo del evento, por caso y versión.

**Cómo leerla.** Con P95, la barra de la clase 5 crece y las de las clases 3 y 4 bajan (expectativas 2 y 4). Sin escalada, el total de municipios con clase ≥ 1 no cambia, porque la rampa es igual por debajo del P90; con suelo saturado puede crecer un poco la clase 2.

In [ ]:
TABLAS = {c: cv.class_counts(C, EXP) for c, C in DATA.items()}
guardar(F.fig_conteos(TABLAS, [cv.label(e) for e in EXP], "municipios"), "conteos_municipios_p99_p95")
guardar(F.fig_conteos(TABLAS, [cv.label(e) for e in EXP], "pixeles_tierra"), "conteos_pixeles_p99_p95")
pd.concat(TABLAS, names=["caso"])

## 6. Prueba ordinal con declaratorias

Municipios dentro de la región de atribución, por clase municipal del máximo del evento, y cuántos tuvieron declaratoria asociada al ciclón (`CT_pico`).

**Cómo leerla.** En cada versión, P(D) debe crecer con la clase. Si P95 rompe el orden (por ejemplo, P(D | clase 5) < P(D | clase 4)), la clase 5 de P95 está mezclando eventos que no corresponden a impactos mayores. Con un solo caso hay pocos municipios por clase (Odile tiene 6 declaratorias): la prueba se lee con todos los casos juntos, al final.

In [ ]:
ORD = []
for c, C in DATA.items():
    inicio = cv.declaratoria_inicio(DECL, cv.CASES[c]["atcf"])
    t = cv.tabla_ordinal(C, set(inicio.index), EXP)
    t.insert(0, "caso", c)
    ORD.append(t)
ORD = pd.concat(ORD, ignore_index=True)
TOTAL = ORD.groupby(["exp", "clase"])[["municipios", "con_declaratoria"]].sum()
TOTAL["P_D"] = TOTAL["con_declaratoria"] / TOTAL["municipios"]
display(TOTAL.unstack("exp").round(3))
ORD.to_csv(FIG / "prueba_ordinal_p99_p95.csv", index=False)

## 7. Para la discusión

Con las expectativas de la introducción, cada punto se responde con una figura:

1. **¿Δclase ≥ 0 en todas partes?** Sección 2. Es una comprobación de construcción.
2. **¿Dónde cambia la clase?** Sección 2 contra sección 3: si los cambios están donde la lluvia pasó del P90, el P95 está haciendo lo que se esperaba.
3. **¿Cuánto crece la clase 5 y cuánto se vacían las clases 3 y 4?** Sección 5.
4. **¿Se mantiene el orden con las declaratorias?** Sección 6.

El Bloque 1 ya mostró, con 381 929 pares municipio–ciclón, que el P95 como máximo no mejora la discriminación (PSS de clase 5: 0.553 → 0.546, intervalo de la diferencia [−0.023, +0.008]) y sube un 61 % las alertas de clase 5. Aquí no se repite esa verificación; se muestra cómo se ve el índice en cinco casos. Adoptar el P95 sería una decisión operativa (alertar más, por el subregistro de las declaratorias y porque los pronósticos subestiman la lluvia), con ese costo en falsas alarmas.